# CheXpert Model Training Pipeline
This notebook consolidates the complete training pipeline including data setup, dataset definition, model architecture, and training loops.

In [1]:
import os
import yaml
import gdown
import zipfile
import glob
import pandas as pd
import numpy as np
import cv2
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision.models import DenseNet121_Weights
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import random
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
from torch.utils.tensorboard import SummaryWriter

In [ ]:
# Configuration
config = {
    'project_name': 'CheXpert_DenseNet_XAI_Final_V3',
    'infrastructure': {
        'seed': 42,
        'device': 'cuda', # "cuda" or "cpu"
        'precision': 'amp'
    },
    'data': {
        'drive_ids': {
            'images': '133K9X8Qrpkcuxm1AGsGr7kh8cQnOAvCk',
            'csv': '1cym7lVa3_ZXZj_-G9gYFNp6AhdogXEIH'
        },
        'paths': {
            'base_dir': './data',
            'images_zip': './data/chexpert_full.zip',
            'csv_zip': './data/csv_splits.zip',
            'extract_dir': './data/extracted_files',
            'model_save_dir': './models',
            'log_dir': './logs'
        },
        'preprocessing': {
            'img_size': [224, 224],
            'clahe_clip': 2.0,
            'rotation_deg': 5,
            'uncertainty_method': 'mask',
            'norm_mean': [0.485, 0.456, 0.406],
            'norm_std': [0.229, 0.224, 0.225]
        },
        'loader': {
            'batch_size': 128,
            'num_workers': 16, # Change according to your environment
            'pin_memory': True
        }
    },
    'model': {
        'backbone': 'densenet121',
        'pretrained': True
    },
    'training': {
        'epochs': 15,
        'early_stopping_patience': 5,
        'skip_to_cooling': False,
        'specialization': {
            'lr': 1e-4,
            'asl_gamma_neg': 4,
            'asl_gamma_pos': 1,
            'asl_clip': 0.05
        },
        'cooling': {
            'lr': 1e-6,
            'epochs': 15,
            'pos_weight_clamp': 10.0
        }
    }
}

In [3]:
def run_provisioning(cfg):
    d_cfg = cfg['data']
    paths = d_cfg['paths']
    
    os.makedirs(paths['base_dir'], exist_ok=True)
    os.makedirs(paths['extract_dir'], exist_ok=True)

    # Download logic using dynamic keys from config
    files_to_download = [
        ('images', d_cfg['drive_ids']['images'], paths['images_zip']),
        ('csv', d_cfg['drive_ids']['csv'], paths['csv_zip'])
    ]

    for name, drive_id, output in files_to_download:
        if not os.path.exists(output):
            print(f"Downloading {name}...")
            gdown.download(f'https://drive.google.com/uc?id={drive_id}', output, quiet=False)
        else:
            print(f"File {name} already exists.")

    # Extraction
    for zip_path in [paths['csv_zip'], paths['images_zip']]:
        if os.path.exists(zip_path):
            print(f"Extracting {zip_path}...")
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(paths['extract_dir'])

# Provisioning data
run_provisioning(config)

File images already exists.
File csv already exists.
Extracting ./data/csv_splits.zip...
Extracting ./data/chexpert_full.zip...


In [4]:
class ApplyCLAHE:
    def __init__(self, clip_limit=2.0):
        self.clip_limit = clip_limit

    def __call__(self, img):
        img_np = np.array(img)
        clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=(8, 8))
        if len(img_np.shape) == 3:
            lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
            l, a, b = cv2.split(lab)
            cl = clahe.apply(l)
            final_img = cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2RGB)
        else:
            final_img = clahe.apply(img_np)
        return Image.fromarray(final_img)

class CheXpertDataset(Dataset):
    def __init__(self, csv_file, extract_dir, cfg, is_train=True):
        self.df = pd.read_csv(csv_file)
        self.extract_dir = extract_dir
        self.is_train = is_train
        
        self.df = self.df.dropna(subset=['absolute_path'])
        
        self.df = self.df.loc[:, ~self.df.columns.str.endswith('.1')]
        
        self.df['clean_path'] = self.df['absolute_path'].apply(
            lambda x: x[x.find('CheXpert-v1.0-small'):] if 'CheXpert-v1.0-small' in str(x) else str(x).lstrip('/')
        )
        
        metadata = ['Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'absolute_path', 'patient_id', 'Patient_ID', 'split', 'unified_key', 'clean_path']
        
        # Enforce numeric type check to prevent any string columns from slipping in
        self.target_cols = [
            c for c in self.df.columns 
            if c not in metadata and pd.api.types.is_numeric_dtype(self.df[c])
        ]
        
        method = cfg['data']['preprocessing']['uncertainty_method']
        
        if method == "delete":
            initial_len = len(self.df)
            has_unc = (self.df[self.target_cols] == -1.0).any(axis=1)
            self.df = self.df[~has_unc].reset_index(drop=True)
            print(f"Uncertainty: Deleted {initial_len - len(self.df)} rows.")
        else:
            self.df[self.target_cols] = self.df[self.target_cols].replace(-1, np.nan)
            print("Uncertainty: Masking methodology applied (NaN transformation).")

        pre = cfg['data']['preprocessing']
        base_tf = [
            transforms.Resize(tuple(pre['img_size'])),
            ApplyCLAHE(clip_limit=pre['clahe_clip']),
            transforms.ToTensor(),
            transforms.Normalize(mean=pre['norm_mean'], std=pre['norm_std'])
        ]
        if self.is_train:
            base_tf.insert(2, transforms.RandomRotation(degrees=pre['rotation_deg']))
        self.transform = transforms.Compose(base_tf)



    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.normpath(os.path.join(self.extract_dir, row['clean_path']))
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        labels = row[self.target_cols].values.astype(np.float32)
        return image, torch.tensor(labels)

def get_dataloaders(cfg):
    ext_dir = cfg['data']['paths']['extract_dir']
    def find_csv(pattern):
        files = glob.glob(os.path.join(ext_dir, "**", pattern), recursive=True)
        if not files: raise FileNotFoundError(f"Missing {pattern}")
        return files[0]

    train_ds = CheXpertDataset(find_csv("Model_Train.csv"), ext_dir, cfg, is_train=True)
    val_ds = CheXpertDataset(find_csv("Model_Val.csv"), ext_dir, cfg, is_train=False)

    l_cfg = cfg['data']['loader']
    train_loader = DataLoader(train_ds, batch_size=l_cfg['batch_size'], shuffle=True, 
                              num_workers=l_cfg['num_workers'], pin_memory=l_cfg['pin_memory'])
    val_loader = DataLoader(val_ds, batch_size=l_cfg['batch_size'], shuffle=False, 
                            num_workers=l_cfg['num_workers'], pin_memory=l_cfg['pin_memory'])
    
    pos_w = []
    max_w = cfg['training']['cooling'].get('pos_weight_clamp', 10.0)
    for col in train_ds.target_cols:
        pos = (train_ds.df[col] == 1.0).sum()
        neg = (train_ds.df[col] == 0.0).sum()
        weight = neg / (pos + 1e-8)
        pos_w.append(np.clip(weight, 0.1, max_w))
        
    return train_loader, val_loader, torch.tensor(pos_w, dtype=torch.float32)

In [5]:
class TaskMaskedASL(nn.Module):
    def __init__(self, config):
        super().__init__()
        spec = config['training']['specialization']
        self.gamma_neg = spec.get('asl_gamma_neg', 4)
        self.gamma_pos = spec.get('asl_gamma_pos', 1)
        self.clip      = spec.get('asl_clip', 0.05)

    def forward(self, x, y):
        mask = ~torch.isnan(y)
        losses = []
        for i in range(x.size(1)):
            x_i, y_i, m_i = x[:, i], y[:, i], mask[:, i]
            x_v, y_v = x_i[m_i], y_i[m_i]
            if x_v.numel() == 0:
                losses.append(x_i.sum() * 0.0)
                continue
            p = torch.sigmoid(x_v)
            n = (1 - p + self.clip).clamp(max=1)
            loss = (
                -y_v * torch.log(p.clamp(min=1e-8)) * (1 - p) ** self.gamma_pos
                - (1 - y_v) * torch.log(n.clamp(min=1e-8)) * p ** self.gamma_neg
            )
            losses.append(loss.mean())
        return losses

def _extract_state_dict(raw):
    if isinstance(raw, dict):
        for key in ("state_dict", "model", "model_state"):
            if key in raw and isinstance(raw[key], dict):
                return raw[key]
    if hasattr(raw, "state_dict"):
        return raw.state_dict()
    return raw

def _sanitize_state_dict(state_dict):
    import re
    fixed = {}
    for k, v in state_dict.items():
        if k.startswith("module."):
            k = k[len("module."): ]
        k = re.sub(r'norm\.(\d+)', r'norm\1', k)
        k = re.sub(r'conv\.(\d+)', r'conv\1', k)
        fixed[k] = v
    return fixed

def _add_features_prefix(state_dict):
    prefixed = {}
    for k, v in state_dict.items():
        if k.startswith(("features.", "classifier.")):
            prefixed[k] = v
        else:
            prefixed[f"features.{k}"] = v
    return prefixed

def _map_classifier_names(state_dict):
    mapped = {}
    for k, v in state_dict.items():
        if k.startswith("fc."):
            mapped[f"classifier.{k[3:]}"] = v
        elif k.startswith("head."):
            mapped[f"classifier.{k[5:]}"] = v
        else:
            mapped[k] = v
    return mapped

def _strip_leading_numeric_segment(state_dict):
    if not state_dict:
        return None
    keys = list(state_dict.keys())
    first_seg = keys[0].split(".", 1)[0]
    if first_seg.isdigit() and all(k.startswith(first_seg + ".") for k in keys):
        return {k[len(first_seg) + 1:]: v for k, v in state_dict.items()}
    return None

def _align_state_dict(state_dict, model_state_dict):
    model_keys = set(model_state_dict.keys())
    if not state_dict:
        return state_dict

    bases = []
    def add_base(sd):
        bases.append(sd)

    add_base(state_dict)
    keys = list(state_dict.keys())

    for prefix in ("module.", "model.", "backbone.", "encoder.", "densenet.", "densenet121."):
        if all(k.startswith(prefix) for k in keys):
            add_base({k[len(prefix):]: v for k, v in state_dict.items()})

    first_seg = keys[0].split(".", 1)[0]
    if all(k.startswith(first_seg + ".") for k in keys):
        add_base({k[len(first_seg) + 1:]: v for k, v in state_dict.items()})

    expanded_bases = []
    for base in bases:
        expanded_bases.append(base)
        stripped = _strip_leading_numeric_segment(base)
        if stripped is not None:
            expanded_bases.append(stripped)

    candidates = []
    def add_candidate(sd):
        candidates.append(sd)

    for base in expanded_bases:
        add_candidate(base)
        add_candidate(_add_features_prefix(base))
        add_candidate(_map_classifier_names(base))

    best = state_dict
    best_overlap = -1
    for cand in candidates:
        overlap = sum(1 for k in cand.keys() if k in model_keys)
        if overlap > best_overlap:
            best_overlap = overlap
            best = cand
    return best

def download_pretrained_weights(save_dir: str) -> str:
    os.makedirs(save_dir, exist_ok=True)

    rad_zip_path = os.path.join(save_dir, "RadImageNet-ResNet50_notop.zip")
    rad_weights_path = os.path.join(save_dir, "RadImageNet_pytorch", "DenseNet121.pt")

    if os.path.exists(rad_weights_path):
        print(f"RadImageNet weights found at: {rad_weights_path}")
        return rad_weights_path

    if os.path.exists(rad_zip_path):
        print(f"Extracting RadImageNet zip: {rad_zip_path}")
        with zipfile.ZipFile(rad_zip_path, 'r') as zip_ref:
            zip_ref.extractall(save_dir)
        if os.path.exists(rad_weights_path):
            print(f"RadImageNet weights extracted to: {rad_weights_path}")
            return rad_weights_path
        print("RadImageNet zip extracted, but DenseNet121.pt not found. Falling back to ImageNet.")

    if not os.path.exists(rad_zip_path):
        print("Downloading RadImageNet weights zip (first-time only)...")
        gdown.download(
            "https://drive.google.com/uc?id=1RHt2GnuOYlc_gcoTETtBDSW73mFyRAtR",
            rad_zip_path,
            quiet=False
        )
        if os.path.exists(rad_zip_path):
            with zipfile.ZipFile(rad_zip_path, 'r') as zip_ref:
                zip_ref.extractall(save_dir)
            if os.path.exists(rad_weights_path):
                print(f"RadImageNet weights extracted to: {rad_weights_path}")
                return rad_weights_path
            print("RadImageNet zip downloaded, but DenseNet121.pt not found. Falling back to ImageNet.")

    weight_path = os.path.join(save_dir, "densenet121_imagenet.pth")

    if os.path.exists(weight_path):
        print(f"ImageNet weights found at: {weight_path}")
        return weight_path

    print("Downloading DenseNet121 ImageNet weights (first-time only)...")
    state_dict = DenseNet121_Weights.IMAGENET1K_V1.get_state_dict(progress=True)
    torch.save(state_dict, weight_path)
    print(f"Weights cached at: {weight_path}")
    return weight_path

def build_model(config, num_classes: int) -> nn.Module:
    m_cfg = config['model']
    paths = config['data']['paths']

    model = models.densenet121(weights=None)

    if m_cfg.get('pretrained', True):
        weight_path = download_pretrained_weights(paths['model_save_dir'])
        raw = torch.load(weight_path, map_location='cpu', weights_only=False)
        state_dict = _extract_state_dict(raw)

        if not isinstance(state_dict, dict):
            raise ValueError(f"Unsupported weights format: {type(state_dict)}")

        fixed = _sanitize_state_dict(state_dict)
        fixed = _align_state_dict(fixed, model.state_dict())
        filtered = {k: v for k, v in fixed.items() if not k.startswith('classifier')}

        missing, unexpected = model.load_state_dict(filtered, strict=False)
        non_classifier_missing = [k for k in missing if not k.startswith('classifier')]
        if non_classifier_missing:
            sample = non_classifier_missing[:10]
            raise RuntimeError(
                f"Backbone keys not loaded: {sample} (missing {len(non_classifier_missing)}). "
                "Check that the weights match DenseNet121."
            )

        print(f"Pretrained backbone loaded: {len(filtered)} keys.")

    num_ftrs = model.classifier.in_features
    model.classifier = nn.Linear(num_ftrs, num_classes)
    print(f"Model ready: DenseNet121 -> {num_classes} output classes.")

    return model

In [6]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def save_checkpoint(model, optimizer, scaler, scheduler, epoch, auc, patience_counter, path, name):
    os.makedirs(path, exist_ok=True)
    state = {
        'epoch':            epoch,
        'model_state':      model.state_dict(),
        'optimizer_state':  optimizer.state_dict(),
        'scaler_state':     scaler.state_dict() if scaler is not None else None,
        'scheduler_state':  scheduler.state_dict() if scheduler is not None else None,
        'best_auc':         auc,
        'patience_counter': patience_counter,
    }
    torch.save(state, os.path.join(path, f"{name}.pth"))
    print(f"--> Checkpoint saved: {name}.pth at Epoch {epoch + 1}")

def load_checkpoint(model, optimizer, scaler, scheduler, path):
    if not os.path.exists(path):
        return 0, 0.0, 0

    print(f"Loading checkpoint from {path}...")
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])

    if scaler is not None and checkpoint.get('scaler_state') is not None:
        scaler.load_state_dict(checkpoint['scaler_state'])
        
    if scheduler is not None and checkpoint.get('scheduler_state') is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state'])

    epoch           = checkpoint['epoch'] + 1
    best_auc        = checkpoint.get('best_auc', 0.0)
    patience_counter = checkpoint.get('patience_counter', 0)

    return epoch, best_auc, patience_counter

class BCEWithPosWeight:
    def __init__(self, pos_weight: torch.Tensor):
        self.pos_weight = pos_weight

def train_one_epoch(model, loader, crit, opt, scaler, device):
    model.train()
    running_loss     = 0.0
    processed_batches = 0

    pbar = tqdm(loader, desc="Training", leave=False)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        mask = ~torch.isnan(labels)

        if not mask.any():
            continue
        processed_batches += 1

        opt.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=device.type,
            enabled=(scaler is not None and device.type == 'cuda')
        ):
            logits = model(imgs)

            # Note: PCGrad logic removed as requested
            if isinstance(crit, TaskMaskedASL):
                losses = crit(logits, labels)
                loss = sum(losses)
            else:
                raw_loss = F.binary_cross_entropy_with_logits(
                    logits,
                    labels.nan_to_num(0),
                    pos_weight=crit.pos_weight,
                    reduction='none'
                )
                loss = raw_loss[mask].mean()
                
            batch_loss = loss.item()

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            opt.step()

        running_loss += batch_loss
        pbar.set_postfix({'loss': f"{batch_loss:.4f}"})

    return running_loss / (processed_batches + 1e-8)

def validate_epoch(model, loader, device):
    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Validation", leave=False):
            imgs   = imgs.to(device)
            logits = model(imgs)
            preds  = torch.sigmoid(logits)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.numpy())

    all_preds  = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    aucs = []
    for i in range(all_labels.shape[1]):
        valid_mask = ~np.isnan(all_labels[:, i])
        if valid_mask.sum() > 0 and len(np.unique(all_labels[valid_mask, i])) == 2:
            auc = roc_auc_score(all_labels[valid_mask, i], all_preds[valid_mask, i])
            aucs.append(auc)

    mean_auc = float(np.mean(aucs)) if aucs else 0.0
    print(f"Validation AUC computed over {len(aucs)} classes.")
    return mean_auc

In [ ]:
def run_training():
    cfg = config
    seed_everything(cfg['infrastructure']['seed'])

    device  = torch.device(cfg['infrastructure']['device'] if torch.cuda.is_available() else "cpu")
    writer  = SummaryWriter(cfg['data']['paths']['log_dir'])
    save_dir = cfg['data']['paths']['model_save_dir']
    patience_limit = int(cfg['training'].get('early_stopping_patience', 7))

    precision = str(cfg['infrastructure'].get('precision', 'amp')).lower()
    use_amp   = (precision == 'amp') and (device.type == 'cuda')

    print(f"Device: {device} | AMP: {use_amp}")

    train_loader, val_loader, pos_weights = get_dataloaders(cfg)
    num_classes = len(train_loader.dataset.target_cols)

    model  = build_model(cfg, num_classes).to(device)
    scaler = torch.amp.GradScaler(enabled=use_amp) if use_amp else None

    spec_latest_path = os.path.join(save_dir, 'spec_checkpoint.pth')
    spec_best_path   = os.path.join(save_dir, 'spec_best_model.pth')
    cool_latest_path = os.path.join(save_dir, 'cooling_checkpoint.pth')
    cool_best_path   = os.path.join(save_dir, 'cooling_best_model.pth')

    def run_specialization():
        print("\n--- Phase 1: Specialization (ASL) ---")
        opt      = torch.optim.Adam(model.parameters(), lr=float(cfg['training']['specialization']['lr']))
        crit_asl = TaskMaskedASL(cfg)
        
        total_epochs = int(cfg['training']['epochs'])
        # Added CosineAnnealingLR
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_epochs)

        start_epoch, best_auc, patience_counter = load_checkpoint(model, opt, scaler, scheduler, spec_latest_path)

        for epoch in range(start_epoch, total_epochs):
            train_loss = train_one_epoch(model, train_loader, crit_asl, opt, scaler, device)
            val_auc    = validate_epoch(model, val_loader, device)
            
            scheduler.step()

            print(f"Spec Epoch {epoch + 1}/{total_epochs} | Loss: {train_loss:.4f} | AUC: {val_auc:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")
            writer.add_scalar('Loss/Spec_Train', train_loss, epoch)
            writer.add_scalar('AUC/Spec_Val',   val_auc,    epoch)
            writer.add_scalar('LR/Spec', scheduler.get_last_lr()[0], epoch)

            if (val_auc > best_auc) or (not os.path.exists(spec_best_path)):
                best_auc        = val_auc
                patience_counter = 0
                save_checkpoint(model, opt, scaler, scheduler, epoch, best_auc, patience_counter, save_dir, 'spec_best_model')
                print(f"*** New Best Specialization AUC: {best_auc:.4f} ***")
            else:
                patience_counter += 1
                print(f"--- No Improvement. Patience: {patience_counter}/{patience_limit} ---")

            save_checkpoint(model, opt, scaler, scheduler, epoch, best_auc, patience_counter, save_dir, 'spec_checkpoint')

            if patience_counter >= patience_limit:
                print(f"Specialization early stopping at Epoch {epoch + 1}.")
                break

    def load_best_spec_weights():
        if os.path.exists(spec_best_path):
            print(f"Loading best specialization weights from {spec_best_path}...")
            ckpt = torch.load(spec_best_path, map_location='cpu', weights_only=False)
            model.load_state_dict(ckpt['model_state'])
            return True
        return False

    def run_cooling():
        print("\n--- Phase 2: Cooling (BCE + per-class pos_weight) ---")
        opt_cool = torch.optim.Adam(
            model.parameters(),
            lr=float(cfg['training']['cooling']['lr'])
        )
        crit_bce = BCEWithPosWeight(pos_weights.to(device))

        total_cooling_epochs = int(cfg['training']['cooling']['epochs'])
        # Added CosineAnnealingLR
        scheduler_cool = torch.optim.lr_scheduler.CosineAnnealingLR(opt_cool, T_max=total_cooling_epochs)

        start_cool_epoch  = 0
        best_cool_auc     = 0.0
        cool_patience_counter = 0

        resume_path = None
        if os.path.exists(cool_latest_path):
            resume_path = cool_latest_path
        elif os.path.exists(cool_best_path):
            resume_path = cool_best_path

        if resume_path is not None:
            start_cool_epoch, best_cool_auc, cool_patience_counter = load_checkpoint(
                model, opt_cool, scaler, scheduler_cool, resume_path
            )
            print(f"Resuming cooling from Epoch {start_cool_epoch + 1} ({os.path.basename(resume_path)}).")
        else:
            loaded = load_best_spec_weights()
            if loaded:
                print("Cooling initialized from best specialization weights.")
            else:
                print("No specialization weights found; cooling starts from current model state.")

        for epoch in range(start_cool_epoch, total_cooling_epochs):
            train_loss = train_one_epoch(model, train_loader, crit_bce, opt_cool, scaler, device)
            val_auc    = validate_epoch(model, val_loader, device)
            
            scheduler_cool.step()

            print(f"Cooling Epoch {epoch + 1}/{total_cooling_epochs} | Loss: {train_loss:.4f} | AUC: {val_auc:.4f} | LR: {scheduler_cool.get_last_lr()[0]:.6f}")
            writer.add_scalar('Loss/Cooling_Train', train_loss, epoch)
            writer.add_scalar('AUC/Cooling_Val',   val_auc,    epoch)
            writer.add_scalar('LR/Cooling', scheduler_cool.get_last_lr()[0], epoch)

            if (val_auc > best_cool_auc) or (not os.path.exists(cool_best_path)):
                best_cool_auc         = val_auc
                cool_patience_counter = 0
                save_checkpoint(model, opt_cool, scaler, scheduler_cool, epoch, best_cool_auc, cool_patience_counter, save_dir, 'cooling_best_model')
                print(f"*** New Best Cooling AUC: {best_cool_auc:.4f} ***")
            else:
                cool_patience_counter += 1
                print(f"--- No Improvement. Patience: {cool_patience_counter}/{patience_limit} ---")

            save_checkpoint(model, opt_cool, scaler, scheduler_cool, epoch, best_cool_auc, cool_patience_counter, save_dir, 'cooling_checkpoint')

            if cool_patience_counter >= patience_limit:
                print(f"Cooling early stopping at Epoch {epoch + 1}.")
                break

    skip_to_cooling = cfg['training'].get('skip_to_cooling', False)
    if os.path.exists(cool_latest_path) or os.path.exists(cool_best_path):
        run_cooling()
    elif skip_to_cooling:
        run_cooling()
    else:
        run_specialization()
        run_cooling()

    writer.close()
    print("Training complete.")

# To run the training pipeline, uncomment the following line:

run_training()

Device: cuda | AMP: True
Uncertainty: Masking methodology applied (NaN transformation).
Uncertainty: Masking methodology applied (NaN transformation).
RadImageNet weights found at: ./models/RadImageNet_pytorch/DenseNet121.pt
Pretrained backbone loaded: 725 keys.
Model ready: DenseNet121 -> 6 output classes.

--- Phase 1: Specialization (ASL) ---
Loading checkpoint from ./models/spec_checkpoint.pth...


Training:   0%|          | 0/1039 [00:00<?, ?it/s]

Validation AUC computed over 6 classes.
Spec Epoch 4/15 | Loss: 0.4689 | AUC: 0.8300 | LR: 0.000083
--> Checkpoint saved: spec_best_model.pth at Epoch 4
*** New Best Specialization AUC: 0.8300 ***
--> Checkpoint saved: spec_checkpoint.pth at Epoch 4


Training:   0%|          | 0/1039 [00:00<?, ?it/s]

Validation:   0%|          | 0/155 [00:00<?, ?it/s]

...

Training:   0%|          | 0/1039 [00:00<?, ?it/s]

Validation:   0%|          | 0/155 [00:00<?, ?it/s]

Validation AUC computed over 6 classes.
Cooling Epoch 1/7 | Loss: 0.2273 | AUC: 0.8397 | LR: 0.000001
--> Checkpoint saved: cooling_best_model.pth at Epoch 1
*** New Best Cooling AUC: 0.8397 ***
--> Checkpoint saved: cooling_checkpoint.pth at Epoch 1


Training:   0%|          | 0/1039 [00:00<?, ?it/s]

Validation:   0%|          | 0/155 [00:00<?, ?it/s]